In [5]:
%pip install delta-spark

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [11]:

import os
print(os.environ.get("JAVA_HOME"))


C:\Program Files\Java\jre1.8.0_461


In [13]:
# Generate synthetic raw data locally with controlled edge cases.
# Usage: python scripts/generate_data.py --seed 42 --out data_raw
import sys
import os
notebook_dir = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(notebook_dir, '..')))
import argparse, os, pathlib, random
import numpy as np
from faker import Faker
from datetime import datetime, timedelta, date
from mimesis import Person, Address
import rstr
import pyarrow as pa
import pyarrow.parquet as pq

#for commerce synthetic data
from faker_commerce import Provider

#timezone check
import pytz

# random character/ digit fix
import string

# Define the correct character set
charset = string.ascii_uppercase + string.digits 

# read csv
import csv

#JSON
import json
import uuid

#xlsx
import pandas as pd

#shipment pq
import pytz

# returns delta
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, LongType, StringType, TimestampType, IntegerType
from delta import configure_spark_with_delta_pip




In [14]:
# Timezone check
# Define the UTC+8 timezone
utc_plus_8 = pytz.timezone('Australia/Perth')  
tz = pytz.timezone('Australia/Perth')  
# Get current date in UTC+8
now = datetime.now(utc_plus_8).date()
nowtime = datetime.now(utc_plus_8)
print("Current date in UTC+8:", now.strftime("%Y-%m-%d %H:%M:%S"))

### DIMENSION TABLES

#Dimension Keys

customer_count = 80000
product_count = 25000
store_count = 5000
supplier_count = 8000
transaction_count = 1000001
transaction_backdate = 200
event_count = 2000001
sensor_count = 1000001
exchangerates_count = 365*3
shipment_count = 1000000
returns_count = 100000


customer_ids = []
product_ids = []
store_ids = []
store_channel_map = {}# for fact table
#    store_code_map = {}
supplier_ids = []
order_ids = []
product_price_map = {} # for fact table

    
# Minimal sample generation (expand to full volumes per docs)
fake = Faker('en_AU')
fake.add_provider(Provider)

Current date in UTC+8: 2025-09-26 00:00:00


In [15]:
# Replace parse_args with manual setup
class Args:
    seed = 42
    out = 'data_raw'

args = Args()

def ensure_dir(p): pathlib.Path(p).mkdir(parents=True, exist_ok=True)

def main():

    random.seed(args.seed)
    np.random.seed(args.seed)
    out = pathlib.Path(args.out)
    ensure_dir(out)

    


    spark = SparkSession.builder \
        .appName("DeltaLakeApp") \
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
        .getOrCreate()




 ### Returns (delta)
    returns_path = out / 'returns.delta'



   # use order_lines as a reference
    # Read into DataFrame    
    orderslines_path = out/'orders_lines.csv'
    df_orders_lines = pd.read_csv(orderslines_path)

  
    #Pre-filter rows with qty ≥ 1
    valid_order_rows = df_orders_lines[df_orders_lines['qty'].fillna(0) >= 1]

    # Prebatch sampling for performance
    sampled_rows = valid_order_rows.sample(n=returns_count).reset_index(drop=True)
    
  # Generate base data
    base_data = []
    for i, row in enumerate(sampled_rows.itertuples(index=False), start=1):
        # Sample a valid row
        qtyret = random.randint(1, int(row.qty))

        # Calculate the time range
        order_ts = pd.to_datetime(row.order_ts)
        time_now = datetime.now(tz)
        time_diff = (time_now- order_ts).total_seconds()

        # Generate a random offset within that range
        random_offset = random.uniform(0, time_diff)

        # Create return_ts
        return_ts = (order_ts + timedelta(seconds=random_offset)).to_pydatetime()  # return_ts cannot be timezone-aware in Spark
        # reason 
        reason = random.choice(["damaged", "wrong item", "changed mind", "late delivery"])
        
        base_data.append((
            i,  # return_id
            row.order_id,  # order_id
            row.product_id, # product_id
            return_ts, # return_ts
            qtyret,  # qty
            reason  # reason
            #,row.order_dt_month # partitioning
        ))
        
    schema_v1 = StructType([
        StructField("return_id", LongType(), False),
        StructField("order_id", LongType(), False),
        StructField("product_id", StringType(), False),
        StructField("return_ts", TimestampType(), False),
        StructField("qty", IntegerType(), False),
        StructField("reason", StringType(), False)
    ])
    # Save as Delta table
    df_returns_v1 = spark.createDataFrame(base_data, schema=schema_v1)
    df_returns_v1.write.format("delta").mode("overwrite").save(str(returns_path))

    

In [10]:
#v2 evolution
    evolved_data = []
    #map evolved schema
    reason_map = {
    "damaged": "DMG",
    "wrong item": "WRONG_IT",
    "changed mind": "CH_MND",
    "late delivery": "LT_DELIV"
    }
    for i in range(returns_count + 1, returns_count + 101):  # 100 new rows to highlight append
        row = df_orders_lines.sample(1).iloc[0]
        order_id = row['order_id']
        order_ts = row['order_ts']

        qtyret = random.randint(1, max(1, int(row['qty'])))
        time_diff = (now - order_ts).total_seconds()
        return_ts = order_ts + timedelta(seconds=random.uniform(0, time_diff))

        reason = random.choice(list(reason_map.keys()))
        reason_code = reason_map[reason]

        evolved_data.append((
            i,
            int(order_id),
            row['product_id'],
            return_ts,
            qtyret,
            reason,
            row['order_dt_month'], # partitioning
            reason_code # added evolution with reasoncode
        ))
    # Updated schema with return_reason_code
    schema_v2 = StructType([
        StructField("return_id", LongType(), False),
        StructField("order_id", LongType(), False),
        StructField("product_id", StringType(), False),
        StructField("return_ts", TimestampType(), False),
        StructField("qty", IntegerType(), False),
        StructField("reason", StringType(), False),
        StructField("return_reason_code", StringType(), True) # evolved schema
    ])

    # Save as Delta table with schema evolution
    df_returns_v2 = spark.createDataFrame(base_data, schema=schema_v2)
    df_returns_v2.write.format("delta").mode("append").option("mergeSchema", "true").save(str(returns_path))
   

IndentationError: unexpected indent (3428300806.py, line 2)

In [16]:
spark = SparkSession.builder.master("local").appName("test").getOrCreate()

KeyboardInterrupt: 

In [7]:
print("creating spark session")
spark = SparkSession.builder.master("local").appName("DeltaLakeTest").config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension").config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog").getOrCreate()
print("spark session created   ")
f_test = spark.createDataFrame([(1, "test")], ["id", "value"])

creating spark session


KeyboardInterrupt: 

In [24]:
builder = SparkSession.builder \
    .appName("DeltaLakeTest") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = SparkSession.builder \
    .appName("DeltaLakeTest") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog").getOrCreate()

# spark = configure_spark_with_delta_pip(builder).getOrCreate()

df_test = spark.createDataFrame([(1, "test")], ["id", "value"])
df_test.write.format("delta").mode("overwrite").save("data_raw/test.delta")


KeyboardInterrupt: 

In [17]:

main()


KeyboardInterrupt: 